# Inference Demo for Mountain NER Model

### Workflow:
1. Load the Hugging Face pipeline, model, and tokenizer.
2. Define a custom function to process pipeline outputs and group consecutive tokens.
3. Run inference on sample texts and display the results with confidence scores.

In [2]:
from transformers import pipeline, AutoModelForTokenClassification, AutoTokenizer

## Initialize the NER pipeline

We set the directory path to the trained model and initialize the token classification pipeline with simple aggregation.

In [3]:
model_dir = "./model"

try:
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForTokenClassification.from_pretrained(model_dir)
except Exception as e:
    print(f"Error loading model from '{model_dir}': {e}")
    tokenizer = AutoTokenizer.from_pretrained("Sava777/mountain_ner")
    model = AutoModelForTokenClassification.from_pretrained("Sava777/mountain_ner")

ner_pipeline = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 1700.03it/s]


## Define the extraction logic

This function takes the raw text, passes it through the pipeline, filters out non-mountain entities, and groups tokens that belong to the same mountain name based on character offsets.

In [4]:
def extract_mountain_names(text, pipeline_instance):
    print(f"\nInput Text: \n> \"{text}\"\n")
    results = pipeline_instance(text)

    if not results:
        print("Result: No mountain names detected in the text.")
        return

    grouped_entities = []
    current_entity = None

    for entity in results:
        label = entity.get("entity_group", entity.get("entity"))

        if label == "O" or not any(target in label for target in ["MOUNTAIN"]):
            continue

        start = entity.get("start")
        end = entity.get("end")
        score = entity.get("score")

        if start is None or end is None:
            continue

        if current_entity is None:
            current_entity = {"start": start, "end": end, "scores": [score]}
        else:
            gap = text[current_entity["end"]:start]
            if gap.strip() == "":
                current_entity["end"] = end
                current_entity["scores"].append(score)
            else:
                grouped_entities.append(current_entity)
                current_entity = {"start": start, "end": end, "scores": [score]}

    if current_entity is not None:
        grouped_entities.append(current_entity)

    if not grouped_entities:
        print("Result: No mountain names detected in the text.")
    else:
        print("Mountain Entities Found:")
        print("-" * 40)
        for entity in grouped_entities:
            word = text[entity["start"]:entity["end"]]
            avg_score = sum(entity["scores"]) / len(entity["scores"])
            print(f" * {word} (Confidence: {avg_score:.1%})")

## Testing the model

We can now pass various sample sentences into our extraction function to test how well the model identifies mountain names in different contexts.

In [6]:
sample_text_1 = "Last summer, my friends and I traveled to Nepal to see Mount Everest, but next year we hope to tackle K2 or Mount Kilimanjaro."
extract_mountain_names(sample_text_1, ner_pipeline)


Input Text: 
> "Last summer, my friends and I traveled to Nepal to see Mount Everest, but next year we hope to tackle K2 or Mount Kilimanjaro."

Mountain Entities Found:
----------------------------------------
 * Mount Everest (Confidence: 98.6%)
 * K2 (Confidence: 88.9%)
 * Mount Kilimanjar (Confidence: 74.5%)


In [5]:
sample_text_2 = "The Andes are the longest continental mountain range in the world, but Denali in Alaska is the highest peak in North America."
extract_mountain_names(sample_text_2, ner_pipeline)


Input Text: 
> "The Andes are the longest continental mountain range in the world, but Denali in Alaska is the highest peak in North America."

Mountain Entities Found:
----------------------------------------
 * Denali (Confidence: 93.2%)


In [8]:
sample_text_3 = "We spent the entire week relaxing on the sunny beaches of California and swimming in the ocean."
extract_mountain_names(sample_text_3, ner_pipeline)


Input Text: 
> "We spent the entire week relaxing on the sunny beaches of California and swimming in the ocean."

Result: No mountain names detected in the text.
